# Truncated Records in Focus Crawl

This notebook is based on: https://github.com/commoncrawl/cc-notebooks/blob/truncation-metrics-2025/warc-truncation/cc-main-2025-truncation-stats.ipynb

The focus crawl uses a content limit of 25 MiB (26214400) as opposed to the 5 MiB (5242880) limit used by the main crawl. This notebooks investigates the impact of this change, i.e., we want to know if we might need to refetch large PDFs similar to what FinePDF did.

Counts of truncated records are aggregated per MIME type from the [columnar index](https://commoncrawl.org/blog/index-to-warc-files-and-urls-in-columnar-format) using [AWS Athena](https://aws.amazon.com/athena/) and the following SQL query (cf. [average-warc-record-length-by-mime-type.sql](https://github.com/commoncrawl/cc-index-table/blob/main/src/sql/examples/cc-index/average-warc-record-length-by-mime-type.sql)):

See also:

- Schema: https://github.com/commoncrawl/cc-index-table/blob/main/src/main/resources/schema/index-schema-simple.json
  - `warc_record_length` (int): Length of the WARC record
  - `content_truncated` (str): Non-null if the WARC record payload is truncated. The value then indicates the reason for the truncation, cf. https://iipc.github.io/warc-specifications/specifications/warc-format/warc-1.1/#warc-truncated

```sql
SELECT COUNT(*) as n_pages,
       COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() as perc_pages,
       AVG(warc_record_length) as avg_warc_record_length,
       SUM(warc_record_length) as sum_warc_record_length,
       MAX(warc_record_length) as max_warc_record_length,
       approx_percentile(warc_record_length, ARRAY[.01,.02,.05,.1,.15,.25,.5,.75,.85,.9,.95,.98,.99,.995])
          as percentiles_warc_record_length,
       SUM(warc_record_length) * 100.0 / SUM(SUM(warc_record_length)) OVER() as perc_warc_storage,
       SUM(case when content_truncated is null then 0 else 1 end) * 100.0 / COUNT(*) as perc_truncated,
       SUM(case when content_truncated is not null then warc_record_length else 0 end)
          as sum_warc_record_length_truncated,
       SUM(case when content_truncated is not null then warc_record_length else 0 end)
          * 100.0 / SUM(SUM(warc_record_length)) OVER() as perc_warc_storage_truncated,
       SUM(case when content_truncated = 'length' then warc_record_length else 0 end)
          as sum_warc_record_length_truncated_length,
       SUM(case when content_truncated = 'length' then warc_record_length else 0 end)
          * 100.0 / SUM(SUM(warc_record_length)) OVER() as perc_warc_storage_truncated_length,
       content_mime_detected,
       histogram(content_truncated) as reason_truncated,
       slice(
         array_sort(
           map_entries(map_filter(
             histogram(regexp_extract(url_path, '\.[a-zA-Z0-9_-]{1,7}$')),
             (k, v) -> v > 4)),
           (a, b) -> IF(a[2] < b[2], 1, IF(a[2] = b[2], 0, -1))),
         1, 25) as common_url_path_suffixes,
       COUNT(DISTINCT url_host_tld) as uniq_tlds,
       approx_distinct(url_host_registered_domain) as uniq_domains,
       approx_distinct(url_host_name) as uniq_hosts,
       slice(
         array_sort(
           map_entries(map_filter(histogram(url_host_tld), (k, v) -> v > 4)),
           (a, b) -> IF(a[2] < b[2], 1, IF(a[2] = b[2], 0, -1))),
         1, 25) as top_tlds,
       approx_most_frequent(25, url_host_registered_domain, 1000) as top_domains
FROM "ccoaindex"."ccoaindex"
WHERE crawl = 'CC-SUPPLEMENTAL-2026-22'
  AND subset = 'warc'
GROUP BY content_mime_detected
HAVING (COUNT(*) >= 10) -- ignore MIME types seen less than 10 times
ORDER BY n_pages DESC;
```

In [1]:
import json
import pandas as pd
from pathlib import Path

# Repo-root-aware data path (works both from repo root and from notebooks/).
DATA_DIR = Path("data") if Path("data").exists() else Path("..") / "data"

df = pd.read_csv(DATA_DIR / 'warc-record-size-truncation-by-mime-type-CC-SUPPLEMENTAL-2026-22.csv')

df[['content_mime_detected', 'n_pages', 'perc_warc_storage',
    'perc_truncated', 'perc_warc_storage_truncated', 'reason_truncated']].head(20)

,content_mime_detected,n_pages,perc_warc_storage,perc_truncated,perc_warc_storage_truncated,reason_truncated
0,text/html,40613284,15.245210,0.002076,0.017853,"{length=255, disconnect=432, time=156}"
1,application/pdf,3169664,73.501854,1.050616,11.550126,"{length=31373, disconnect=1713, time=215}"
2,application/xhtml+xml,2417807,0.541387,0.001324,0.004805,"{length=19, disconnect=6, time=7}"
3,text/plain,875344,1.398873,0.570404,0.482635,"{length=4923, disconnect=54, time=16}"
4,application/x-bibtex-text-file,147997,0.003215,0.000000,0.000000,NaN
5,application/xml,108643,0.035393,0.138067,0.005305,{length=150}
6,application/atom+xml,89137,0.003671,0.000000,0.000000,NaN
7,text/calendar,87210,0.002174,0.000000,0.000000,NaN
8,application/octet-stream,81950,6.007911,16.019524,3.543030,"{length=12744, disconnect=383, time=1}"
9,application/x-pds,80351,0.012730,0.002489,0.000368,{length=2}


The aggregations show which MIME types are mostly affected by truncations.

Now let's look into the reasons of the truncation and load the histograms with reason counts into columns:

In [2]:
# expand embedded Presto/Trino/Athena histogram as columns into data frame
def unroll_histograms(df):
    # - transform to valid JSON
    df['reasons_truncation'] = df['reason_truncated'].str.replace('(\\w+)=', '"\\1":', regex=True)
    df['top_domains'] = df['top_domains'].str.replace('([a-z0-9.-]+)=', '"\\1":', regex=True)

    # - load columns in data frame
    truncation_reason = df['reasons_truncation'].apply(
        lambda x: json.loads(x) if type(x) == str else {}
    ).apply(pd.Series).apply(lambda s: s.fillna(0).astype(int)).add_prefix('trunc_reason_')

    # - join with original data
    df = df.join(truncation_reason)

    df['n_pages_truncated'] \
        = df['trunc_reason_length'] + df['trunc_reason_time'] + df['trunc_reason_disconnect']
    df['trunc_reason_length_perc'] = 100.0 * df['trunc_reason_length'] / df['n_pages']
    df['trunc_length_gib'] = df['sum_warc_record_length_truncated_length'] / 2**30

    return df

df = unroll_histograms(df)
df[['content_mime_detected', 'n_pages', 'perc_truncated', 'n_pages_truncated',
      'trunc_reason_length', 'trunc_reason_length_perc', 'trunc_length_gib']
    ].sort_values(by=['trunc_reason_length'], ascending=False).head(20)

,content_mime_detected,n_pages,perc_truncated,n_pages_truncated,trunc_reason_length,trunc_reason_length_perc,trunc_length_gib
1,application/pdf,3169664,1.050616,33301,31373,0.989789,712.754483
8,application/octet-stream,81950,16.019524,13128,12744,15.550946,220.814640
3,text/plain,875344,0.570404,4993,4923,0.562407,30.020258
24,model/vnd.valve.source.compiled-map,9313,35.359175,3293,3293,35.359175,50.344730
0,text/html,40613284,0.002076,843,255,0.000628,1.058045
55,chemical/x-cache,1505,16.677741,251,251,16.677741,0.042768
95,chemical/x-cif,256,87.890625,225,225,87.890625,3.743903
42,text/troff,2279,7.810443,178,178,7.810443,3.789980
99,application/x-sqlite3,214,76.168224,163,161,75.233645,1.099544
68,application/vnd.ms-pki.stl,687,22.125182,152,152,22.125182,1.677405


## TLDs and Domains with Truncated Content

These statistics try to uncover any oddities in the distributions of truncated and overlong pages/documents over top-level and pay-level domains.

```sql
SELECT COUNT(*) as n_pages,
       content_mime_detected,
       slice(
         array_sort(
           map_entries(map_filter(histogram(url_host_tld), (k, v) -> v > 4)),
           (a, b) -> IF(a[2] < b[2], 1, IF(a[2] = b[2], 0, -1))),
         1, 25) as top_tlds,
       approx_most_frequent(50, url_host_registered_domain, 2000) as top_domains
FROM "ccoaindex"."ccoaindex"
WHERE crawl = 'CC-SUPPLEMENTAL-2026-22'
  AND subset = 'warc'
  AND content_truncated = 'length'
GROUP BY content_mime_detected
HAVING (COUNT(*) >= 10) -- ignore MIME types seen less than 10 times
ORDER BY n_pages DESC;
```

In [3]:
from collections import Counter

def compare_top_domains(df_all, df_trunc, mime='text/html', n=25):
    da = pd.DataFrame(
                Counter(df_all.loc[df_all['content_mime_detected'] == mime,
                                          'top_domains'].apply(json.loads).values[0]).most_common(),
                columns = ['domain', 'count_all']
    )
    dt = pd.DataFrame(
                Counter(df_trunc.loc[df_trunc['content_mime_detected'] == mime,
                                              'top_domains'].apply(json.loads).values[0]).most_common(),
                columns = ['domain', 'count_trunc']
    )
    d = dt.merge(da, how = 'outer').fillna(0).astype({'count_all': int, 'count_trunc': int})
    d['%'] = 100.0 * d['count_trunc'] / d['count_all']
    print(mime)
    return d.head(n)


df_trunc = pd.read_csv(DATA_DIR / 'warc-truncation-domains-CC-SUPPLEMENTAL-2026-22.csv')
df_trunc['top_domains'] = df_trunc['top_domains'].str.replace('([a-z0-9.-]+)=', '"\\1":', regex=True)


compare_top_domains(df, df_trunc, mime='text/html')

text/html


,domain,count_trunc,count_all,%
0,apache.org,0,132279,0.000000
1,arcgis.com,0,152254,0.000000
2,astrouw.edu.pl,18,0,inf
3,bcgsc.ca,2,0,inf
4,berkeley.edu,0,138929,0.000000
5,birmingham.ac.uk,1,0,inf
6,brenda-enzymes.org,19,0,inf
7,caltech.edu,0,190863,0.000000
8,canada.ca,0,138984,0.000000
9,cern.ch,0,148213,0.000000


**Note**: for the MIME type "text/html" there are certain domains which might require a closer look because the rate of truncated docs is high.

In [4]:
compare_top_domains(df, df_trunc, mime='application/xhtml+xml')

application/xhtml+xml


,domain,count_trunc,count_all,%
0,arlis.org,0,69653,0.0
1,bgs.ac.uk,0,79214,0.0
2,blogspot.com,0,93519,0.0
3,clemson.edu,1,0,inf
4,dockflow.org,1,0,inf
5,edge.org,0,30826,0.0
6,esf.edu,1,0,inf
7,harvard.edu,0,36310,0.0
8,ikonet.com,0,43195,0.0
9,jmp.com,0,58606,0.0


In [5]:
compare_top_domains(df, df_trunc, mime='application/pdf')

application/pdf


,domain,count_trunc,count_all,%
0,academie-sciences.fr,0,19277,0.000000
1,admin.ch,232,0,inf
2,arlis.org,417,0,inf
3,auburn.edu,408,0,inf
4,bayern.de,465,0,inf
5,berkeley.edu,201,0,inf
6,bom.gov.au,0,26922,0.000000
7,bournemouth.ac.uk,281,0,inf
8,caltech.edu,595,0,inf
9,cbd.int,0,19612,0.000000


In [6]:
def compare_top_tlds(df_all, df_trunc, mime='text/html', n=25):
    da = dict()
    [da.update(i) for i in df_all.loc[df_all['content_mime_detected'] == mime,
                                             'top_tlds'].apply(json.loads).values[0]]
    da = pd.DataFrame(da.items(), columns = ['tld', 'count_all'])
    dt = dict()
    [dt.update(i) for i in df_trunc.loc[df_trunc['content_mime_detected'] == mime,
                                                 'top_tlds'].apply(json.loads).values[0]]
    dt = pd.DataFrame(dt.items(), columns = ['tld', 'count_trunc'])
    d = dt.merge(da, how = 'outer').fillna(0).astype({'count_all': int, 'count_trunc': int})
    d['%'] = 100.0 * d['count_trunc'] / d['count_all']
    print(mime)
    return d.head(n)

df['top_tlds'] = df['top_tlds'].str.replace('([a-z0-9-]+),', '"\\1":', regex=True)
df_trunc['top_tlds'] = df_trunc['top_tlds'].str.replace('([a-z0-9-]+),', '"\\1":', regex=True)

compare_top_tlds(df, df_trunc, mime='text/html')

text/html


,tld,count_trunc,count_all,%
0,at,0,305854,0.000000
1,au,0,472364,0.000000
2,br,0,279125,0.000000
3,ca,0,754304,0.000000
4,ch,0,537122,0.000000
5,cn,0,277346,0.000000
6,com,50,8204849,0.000609
7,de,0,2390003,0.000000
8,edu,53,4364031,0.001214
9,es,0,490055,0.000000


In [7]:
compare_top_tlds(df, df_trunc, mime='application/xhtml+xml')

application/xhtml+xml


,tld,count_trunc,count_all,%
0,ar,0,22286,0.000000
1,at,0,8763,0.000000
2,au,0,12842,0.000000
3,be,0,74858,0.000000
4,br,0,48024,0.000000
5,ca,6,63671,0.009423
6,cern,0,51238,0.000000
7,ch,0,61890,0.000000
8,cn,0,26630,0.000000
9,com,0,529973,0.000000


In [8]:
compare_top_tlds(df, df_trunc, mime='application/pdf')

application/pdf


,tld,count_trunc,count_all,%
0,at,363,34008,1.067396
1,au,0,41305,0.000000
2,br,0,45249,0.000000
3,ca,2730,77882,3.505303
4,cc,1016,81456,1.247299
5,ch,879,72648,1.209944
6,com,1252,176972,0.707457
7,cz,273,0,inf
8,de,2435,329161,0.739760
9,dk,277,0,inf


### A Closer Look on Domains With Large Amounts of Truncated Content

The aim is to identify domains which will "profiteers" if the content limit threshold is increased - domains which already (with the current 1 MiB limit) have many truncated captures or occupy a large amount of WARC storage.

Get reliable metrics on the level of registered "pay-level" domains
- number of truncated pages
- compared to / ratio of all page captures
- GiB truncated content in WARC files

```sql
with tmp1 as (select
  count(*) as count,
  content_mime_detected,
  url_host_registered_domain
from "ccoaindex"."ccoaindex"
where crawl = 'CC-SUPPLEMENTAL-2026-22'
  and subset = 'warc'
group by content_mime_detected,
  url_host_registered_domain),
tmp2 as (select
  count(*) as count,
  sum(warc_record_length) as sum_warc_record_length,
  content_mime_detected,
  url_host_registered_domain
from "ccoaindex"."ccoaindex"
where crawl = 'CC-SUPPLEMENTAL-2026-22'
  and subset = 'warc'
  and content_truncated = 'length'
group by content_mime_detected,
  url_host_registered_domain
having count(*) >= 10000
    or sum(warc_record_length) > 10*1024*1024*cast(1024 as bigint))
select trunc.count as count_trunc,
  all.count as count_total,
  format('%.3f', 100.0 * trunc.count / all.count) as perc_trunc,
  format('%,.2f', trunc.sum_warc_record_length / (1024.0*1024*1024)) storage_trunc_gib,
  all.content_mime_detected as content_mime_detected,
  all.url_host_registered_domain as url_host_registered_domain
from tmp1 as all
 right outer join tmp2 as trunc
 on all.content_mime_detected = trunc.content_mime_detected
and all.url_host_registered_domain = trunc.url_host_registered_domain
order by trunc.sum_warc_record_length desc;
```

In [9]:
domains = pd.read_csv(DATA_DIR / 'warc-truncation-domains-detailed-CC-SUPPLEMENTAL-2026-22.csv')
domains.head(50)

,count_trunc,count_total,perc_trunc,storage_trunc_gib,content_mime_detected,url_host_registered_domain
0,4294,25907,16.575,55.71,application/octet-stream,nasa.gov
1,3291,9311,35.345,50.30,model/vnd.valve.source.compiled-map,nasa.gov
2,2137,3198,66.823,47.88,application/octet-stream,bcgsc.ca
3,1502,4298,34.946,30.71,application/octet-stream,ucsc.edu
4,1265,17132,7.384,27.65,application/pdf,uqam.ca
5,1113,10964,10.151,25.71,application/pdf,scholaris.ca
6,936,7582,12.345,22.45,application/pdf,ipums.org
7,1623,9502,17.081,21.24,application/octet-stream,mit.edu
8,798,2204,36.207,18.39,application/octet-stream,salmobase.org
9,717,3282,21.846,16.65,application/octet-stream,wisc.edu


## Summary

Headline: PDFs dominate everything.

- application/pdf is only ~6% of records by count (3.17 M of ~47 M) but 73.5 % of total WARC storage. Truncation analysis is effectively a PDF question.
- ~1.05 % of PDFs are truncated (33,301 records), and 11.55 % of all WARC storage is truncated PDFs — about 712.75 GiB of payload that hit the cap. Of those 33,301 truncated PDFs, 31,373 are length-truncated
(i.e. the 25 MiB limit, not a connection drop or timeout). So nearly all PDF truncation is "the document was bigger than 25 MiB."

Other MIMEs worth noting.

- application/octet-stream: 16 % truncation rate, ~221 GiB length-truncated. Heavily driven by nasa.gov, bcgsc.ca, ucsc.edu, mit.edu, salmobase.org, wisc.edu — looks like bioinformatics/science data dumps.
- model/vnd.valve.source.compiled-map: 35 % truncated, 50 GiB, all from nasa.gov. Niche but striking.
- text/html: essentially unaffected (0.002 % truncated, 843 records total, mostly disconnects/timeouts, only 255 by length, ~1 GiB). HTML fits in 25 MiB just fine.
- Long tail of high-rate-but-low-volume types: chemical/x-cif (88 %), application/x-sqlite3 (76 %), application/vnd.apple.keynote (28 %) — interesting curiosities, not material to total losses.

Where the loss is concentrated. The detailed per-domain table makes the targeting obvious. PDF length-truncations cluster on academic / government / research repositories:

- uqam.ca 27.6 GiB, scholaris.ca 25.7 GiB, ipums.org 22.5 GiB, neurips.cc 14.7 GiB, eartharxiv.org 14.5 GiB, columbia.edu, caltech.edu, jaea.go.jp, bayern.de, nasa.gov, uni-bayreuth.de, umk.pl.
- TLD breakdown for PDFs confirms it: .ca 3.5 %, .nl 1.85 %, .edu 1.43 %, .gov 1.31 %, .jp 1.15 % — all well above the .com baseline of 0.7 %. So the truncated content is dominated by long research PDFs, theses,
datasets, conference proceedings.